# Kaggle – Data on the top
Tu profe ha decidido cambiar de aires y, por eso, ha comprado una tienda de portátiles. Sin embargo, su única especialidad es Data Science, por lo que ha decidido crear un modelo de ML para establecer los mejores precios.

¿Podrías ayudar a tu profe a mejorar ese modelo?

## Métrica: RMSE

$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

Donde $y_i$ es el valor real y $\hat{y}_i$ es el valor predicho. **Cuanto menor, mejor.**

---
# PARTE 1: Entrenamiento del modelo

## 1. Librerías

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from toolbox_ml.eda.core import tipifica_variables, describe_df, verificar_dataframe

## 2. Datos

In [3]:
df = pd.read_csv('./data/train.csv', encoding='latin-1')

### 2.1 Exploración de los datos

In [4]:
# Tu código aquí
df.head()

,laptop_ID,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_in_euros
0,755,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,8GB,256GB SSD,Intel HD Graphics 520,Windows 10,1.86kg,539.00
1,618,Dell,Inspiron 7559,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,16GB,1TB HDD,Nvidia GeForce GTX 960<U+039C>,Windows 10,2.59kg,879.01
2,909,HP,ProBook 450,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8GB,1TB HDD,Nvidia GeForce 930MX,Windows 10,2.04kg,900.00
3,2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
4,286,Dell,Inspiron 3567,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2.0GHz,4GB,1TB HDD,AMD Radeon R5 M430,Linux,2.25kg,428.00


In [5]:
df.describe()

,laptop_ID,Inches,Price_in_euros
count,912.000000,912.000000,912.000000
mean,650.312500,14.981579,1111.724090
std,382.727748,1.436719,687.959172
min,2.000000,10.100000,174.000000
25%,324.750000,14.000000,589.000000
50%,636.500000,15.600000,978.000000
75%,982.250000,15.600000,1483.942500
max,1320.000000,18.400000,6099.000000


In [7]:
describe_df(df)

,tipo,porcentaje_nulos,valores_unicos,porcentaje_cardinalidad
laptop_ID,int64,0.0,912,100.00
Company,str,0.0,19,2.08
Product,str,0.0,480,52.63
TypeName,str,0.0,6,0.66
Inches,float64,0.0,17,1.86
ScreenResolution,str,0.0,36,3.95
Cpu,str,0.0,107,11.73
Ram,str,0.0,9,0.99
Memory,str,0.0,37,4.06
Gpu,str,0.0,93,10.20


In [ ]:
# Función de nuestro toolkit!
# Que no esté vacío
# Que no tenga duplicados
# Que no tenga nulos
# Que no tenga infinitos ni columnas constantes
verificar_dataframe(df)

The DataFrame is ready for modeling


## Estudiadas las columnas por encima (con DataWranggler)
### Procedimiento con cada una:

1. Laptop_ID: La tiro. Cada fila es única y no aporta nada (son IDs). La tendré que volver a poner al final en el test.csv porque no me dejará hacer submission o saldrá mal el score, pero la tiro.
2. Company: Se queda. Categórica de manual. Dentro del pipeline (trabajaré con él, ya no puedo "desverlo") haré un one-hot.
3. Product: En principio, la tiro. Parece una tontería, pero me explico: Las características son las que hacen al producto, no al revés. Estaría prediciendo un nombre. Aun así, probaría en algún momento a incluirla por si acaso, pero en principio no tiene cabida.
4. TypeName: Se queda. Categórica de seis valores. One-hot también en el pipeline.
5. Inches: Numérica continua. Se queda igual, sin cambios. La convertiría en categórica, pero perdería ese peso que tienen las numéricas. A mayor tamaño, mayor es el precio. Se queda así.
6. ScreenResolution: Hay que trabajar en esta. Las resoluciones me gustaría tenerlas separadas de manera numérica. Quizá haga directamente la operación de la resolución (multiplicar). Después, toca crear dos columnas binarias de 0s y 1s de IPS y Touchscreen. Pueden no tener ninguno, tener alguno de los dos o tener los dos. Queda fuera la definición de la resolución (full HD por ejemplo) por ser redundante.
7. CPU: Hay que trabajar también. Los GHZ sí o sí como columna numérica. La marca como categórica. Y luego la gama (i3, i5...) que me gustaría saber cómo categorizarlas. Quisiera hacer un ordinal encoder a mano, pero no sé bien cómo organizarlo al haber varias marcas. Lo estudiaré en su debido momento.
8. RAM: Le quito el GB y queda numérica.
9. Memory: Toca trabajar un poco con esta. Separaré en columnas numéricas del tipo: SSD_GB, HDD_GB, Flash_GB, Hybrid_GB. Pasar TB a GB para unificar todo en la misma escala. Los casos con múltiples discos duros, quedarán las dos columnas llenas: SSD_GB=256 y HDD_GB=1024. Que el modelo aprenda solito que SSD encarece más.
10. GPU: Saco la marca (intel, nvidia, AMD) y convierto eso en categórica. Las de intel serán las más baratas y luego tengo Nvidia y AMD. Lástima no tener los GB, pero tengo los modelos. Puedo trabajar en sus gamas (GTX, RTX, R5, R7...) igual que en CPU.
11. OpSys: Categórica. En el DataWranggler pude ver que hay un macOS y un MacOS X. Unifico a macOS que es el mismo sistema, pero el X es como se llamaba antes.
12. Weight: Le quito el KG y queda en numérica continua. Limpia.
13. Price_in_euros: Target.

In [ ]:
# Como vimos en el ejemplo de clase, vamos a tener que hacer el mismo proceso dos veces
# Eso significa que lo mejor es hacerme una bonita función

def df_tc_preparado(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop(columns=["laptop_ID", "Product"])
    df["Ram"] = df["Ram"].str.replace("GB", "").astype(int)
    df["OpSys"] = df["OpSys"].replace({"Mac OS X": "macOS"})
    df["Weight"] = df["Weight"].str.replace("kg", "").astype(float)

    # Las dos columnas nuevas a extraer para meterlas como categorías
    df["IPS"] = df["ScreenResolution"].str.contains("IPS", case=False).astype(int)
    df["TouchScreen"] = df["ScreenResolution"].str.contains("TouchScreen", case=False).astype(int)

    # Ahora las columnas de la resolución.
    # Como no encontraba manera de extraer fácil la resolución con pandas/python, he tenido que usar magia prohibida:
    # Expresiones regulares.
    resolucion = df["ScreenResolution"].str.extract(r"(\d+)x(\d+)")
    # Después de este dolor de cabeza, extraigo ancho y alto (1920 y 1080, por ejemplo)
    ancho = resolucion[0].astype(int)
    alto = resolucion[1].astype(int)
    # Y me hago una columna nueva con los píxeles totales. En escala, esta columna representa un peso numérico real
    # Porque a mayor el número, mayor el precio.
    df["pixeles_totales"] = ancho * alto
    df = df.drop(columns=["ScreenResolution"])

    # CPU
    # Extraer primero la marca y luego los GHz
    # La marca es la primera palabra
    df["CPU_Marca"] = df["Cpu"].str.split().str[0]
    # El reloj es la última palabra y además hay que quitarle GHz para que quede numérica
    df["CPU_GHz"] = df["Cpu"].str.split().str[-1]
    df["CPU_GHz"] = df["CPU_GHz"].str.replace("GHz", "").astype(float)
    # Extraigo la gama:
    df["CPU_Gama"] = df["Cpu"].apply(extraer_gama_cpu)


    # GPU
    # Extraer primero la marca (es la primera palabra, sencillo)
    df["GPU_Marca"] = df["Gpu"].str.split().str[0]

    return df


# La función que va dentro de df_tc_preparado()
# Es para extraer la gama del CPU a mano
# No se me ocurre otra manera
def extraer_gama_cpu(texto_cpu):
    # Paso a minúsculas
    texto = texto_cpu.lower()

    # Extraigo las de intel
    if "i7" in texto:
        return "Core i7"
    elif "i5" in texto:
        return "Core i5"
    elif "i3" in texto:
        return "Core i3"
    elif "i9" in texto:
        return "Core i9"
    elif "core m" in texto:
        return "Core M"
    elif "xeon" in texto:
        return "Xeon"
    # Intel gama baja
    elif "celeron" in texto:
        return "Celeron"
    elif "pentium" in texto:
        return "Pentium"
    elif "atom" in texto:
        return "Atom"
    # Extraigo AMD (lo estoy odiando mucho por no tener una estructura tan clara como intel)
    elif "ryzen" in texto:
        return "AMD Ryzen"
    elif "e-series" in texto:
        return "AMD E-Series"
    elif "-series" in texto: # Había separado los números, pero es tontería. Reduzco a A-Series
        return "AMD A-Series"
    elif "fx" in texto:
        return "AMD FX"
    # Todo lo demás.
    # TODO: Revisar si se me escapa alguna gama importante
    else:
        return "Otro"

In [15]:
df["CPU_GHz"] = df["Cpu"].str.split().str[-1]
df["CPU_GHz"] = df["CPU_GHz"].str.replace("GHz", "").astype(float)

df["CPU_GHz"].value_counts()

CPU_GHz
2.50    199
2.80    120
2.70    117
1.60    107
2.00     60
2.30     58
1.80     51
2.60     47
1.10     37
2.40     33
2.90     15
3.00     12
1.20     10
1.44      9
1.50      9
2.20      7
1.30      5
3.60      4
0.90      3
1.90      2
3.10      2
2.10      2
1.00      1
3.20      1
1.92      1
Name: count, dtype: int64

### 2.2 Definir X e y


In [ ]:
# Tu código aquí


### 2.3 Dividir en train y test

In [ ]:
# Tu código aquí


## 3. Procesado de datos

> 🚨 **Data leakage:** si usas un scaler, haz **`.fit()` SOLO sobre `X_train`** y luego aplica `.transform()` sobre `X_train` y `X_test` por separado.
>
> Recuerda también que **todo lo que hagas aquí deberás replicarlo después en `test.csv`** (sección 6).

In [ ]:
# Tu código aquí


## 4. Modelado

### 4.1 Entrenamiento

In [ ]:
# Tu código aquí


### 4.2 Métricas

Recuerda que en la competición se evalúa con **RMSE**.

In [ ]:
# Tu código aquí


### 4.3 Optimización (up to you 🫰🏻)

In [ ]:
# Tu código aquí


## 5. Reentrenamiento sobre todos los datos de `train.csv`

Una vez afinado el modelo, reentrenamos con **todos** los datos disponibles antes de predecir sobre `test.csv`.

> ¿Por qué? El split anterior era solo para validar localmente. Para la submission final queremos aprovechar el 100% de los datos de entrenamiento.

In [ ]:
# Tu código aquí


---
# PARTE 2: Predicción y submission

Una vez tengas el modelo listo, toca predecir sobre `test.csv` y generar el archivo de submission.

## 6. Carga los datos de `test.csv`

In [ ]:
X_pred = pd.read_csv('./data/test.csv', encoding='latin-1')
X_pred.head()

## 7. Replica el procesado en `test.csv`

> ⚠️ Usa `.transform()`, **nunca `.fit_transform()`** sobre los datos de test.
>
> Lo único que **no puedes hacer** es eliminar filas.

In [ ]:
# Tu código aquí


## 8. Genera la submission

### 8.1 ¿Qué formato espera Kaggle?

In [ ]:
sample = pd.read_csv('./data/sample_submission.csv', encoding='latin-1')
sample.head()

### 8.2 Crea tu submission

In [ ]:
# Tu código aquí


### 8.3 Chequeador

Pásale el chequeador antes de subir a Kaggle. Si todo está bien, guardará el CSV automáticamente con un nombre único.

In [ ]:
def checker(df_to_submit, sample, filename=None):
    """
    Valida que tu submission tenga la forma requerida por Kaggle.
    Si es correcta, guarda el CSV listo para subir.
    Si no, lee el mensaje de error y corrígelo.
    """
    if df_to_submit.shape != sample.shape:
        print(' Shape incorrecto.')
        print(f'   Tu submission: {df_to_submit.shape} | Esperado: {sample.shape}')
        print('   Revisa que no hayas borrado filas del test ni añadido/quitado columnas.')
        return

    if not (df_to_submit.columns == sample.columns).all():
        print(' Nombres de columnas incorrectos.')
        print(f'   Tus columnas:       {list(df_to_submit.columns)}')
        print(f'   Columnas esperadas: {list(sample.columns)}')
        return

    if not (df_to_submit['laptop_ID'] == sample['laptop_ID']).all():
        print(' Los IDs no coinciden con sample_submission. Revisa que no hayas reordenado el test.csv.')
        return

    if filename is None:
        from datetime import datetime
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f'submission_{timestamp}.csv'

    df_to_submit.to_csv(filename, index=False)
    print(f" ¡Todo correcto! Submission guardada como '{filename}'. ¡A Kaggle!")

In [ ]:
checker(submission, sample)